# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** 43  
**Kaggle challenge:** `Deep learning` (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "Choco Hunters"  

**Author 1 (sciper):** Ewa Miazga (367059)  
**Author 2 (sciper):** Sameh Lahouar (300454)   
**Author 3 (sciper):** Nour Guermazi (314474) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

## 00. Imports

In [1]:
from loader import CustomIAPRDataloader, CocoDataset, UnlabeledImageFolder
from models.cnn import SimpleCNN
from models.mobile import LightFasterRCNNMobileNetV3
from helper import get_device, compute_mean_std
from trainer import Trainer
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn
import os 
import cv2
import shutil
import tqdm
import numpy as np
import json
import pandas as pd 
import torchvision.transforms as T

device = get_device()
print(f"Using device: {device}")

Using device: mps


## 00.1 Preprocess dataset with coco

In [8]:
def extract_patch(image_path, bbox, size=1400):
    x, y, w, h = [int(coord) for coord in bbox]
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Image not found: {image_path}")
    patch = img[y:y+h, x:x+w]
    return patch

def patches_from_coco(source, dest, test_mode=False):
    """
    If test_mode is True, extracts full-image patches from each image in the folder.
    Otherwise, uses COCO annotations to extract object patches with labels.
    """
    patches_dir = os.path.join(dest, "patches")

    if os.path.exists(patches_dir):
        shutil.rmtree(patches_dir)
    os.makedirs(patches_dir)

    if test_mode:
        print("🔍 Running in TEST mode (no annotations)...")

        image_files = [f for f in os.listdir(source) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
        for i, image_file in tqdm(enumerate(image_files), total=len(image_files)):
            image_path = os.path.join(source, image_file)
            try:
                img = cv2.imread(image_path)
                if img is None:
                    raise ValueError("Image not found or unreadable")
                idx = str(i).zfill(3)
                patch_path = os.path.join(patches_dir, f"{idx}.jpg")
                cv2.imwrite(patch_path, img)
            except Exception as e:
                print(f"⚠️ Skipped image {image_file}: {e}")

        print(f"✅ Saved {len(image_files)} full-image patches to {patches_dir}")

    else:
        print("🧠 Running in TRAIN mode (with annotations)...")

        annotations_file = os.path.join(source, "_annotations.coco.json")
        images_dir = source
        patches_dir = os.path.join(dest, "patches")

        if os.path.exists(patches_dir):
            shutil.rmtree(patches_dir)
        os.makedirs(patches_dir)

        data = json.load(open(annotations_file, "r"))

        id_to_label = {e["id"]: e["name"] for e in data["categories"]}
        id_to_images = {e["id"]: e["file_name"] for e in data["images"]}
        annotations = data["annotations"]

        df_labels = pd.DataFrame(columns=["name", "label", "image", "bbox"])

        for i, annotation in tqdm(enumerate(annotations), total=len(annotations)):
            image_id = annotation["image_id"]
            label_id = annotation["category_id"]
            bbox = annotation["bbox"]
            label = id_to_label[label_id]
            image_file = id_to_images[image_id]
            image_path = os.path.join(images_dir, image_file)

            try:
                patch = extract_patch(image_path, bbox)
                idx = str(i).zfill(3)
                patch_path = os.path.join(patches_dir, f"{idx}.jpg")
                cv2.imwrite(patch_path, patch)
                df_labels.loc[i] = [idx, label, image_file, bbox]
            except Exception as e:
                print(f"⚠️ Skipped patch {i} from image {image_file}: {e}")

        df_labels.to_csv(os.path.join(patches_dir, "labels.csv"), index=False)
        print(f"✅ Saved {len(df_labels)} patches and labels to {patches_dir}")

# Cell 2: Define paths and load annotation info
train_img_dir = "dataset_project_iapr2025_coco/train_annotated"
ann_path = os.path.join(train_img_dir, "_annotations.coco.json")

# Load COCO annotations and count classes
with open(ann_path, "r") as f:
    coco_json = json.load(f)
categories = coco_json["categories"]
num_classes = len(categories) + 1  # +1 for background
print("Detected classes:", [c["name"] for c in categories])
print("Total (with background):", num_classes)

# Run it on your dataset
source_path = "dataset_project_iapr2025_coco/train_annotated"
destination_path = "dataset_project_iapr2025_coco/train_patches"
patches_from_coco(source_path, destination_path)

Detected classes: ['objects', 'Amandina', 'Arabia', 'Comtesse', 'Creme_brulee', 'Jelly_Black', 'Jelly_Milk', 'Jelly_White', 'Noblesse', 'Noir_authentique', 'Passion_au_lait', 'Stracciatella', 'Tentation_noir', 'Triangolo']
Total (with background): 15
🧠 Running in TRAIN mode (with annotations)...


100%|██████████| 584/584 [00:20<00:00, 29.12it/s]

✅ Saved 584 patches and labels to dataset_project_iapr2025_coco/train_patches/patches


In [ ]:
# Cell 2: Define paths and load annotation info
train_img_dir = "dataset_project_iapr2025-2/train_annotated"
ann_path = os.path.join(train_img_dir, "_annotations.coco.json")

# Load COCO annotations and count classes
with open(ann_path, "r") as f:
    coco_json = json.load(f)
categories = coco_json["categories"]
num_classes = len(categories) + 1  # +1 for background
print("Detected classes:", [c["name"] for c in categories])
print("Total (with background):", num_classes)

# Run it on your dataset
source_path = "dataset_project_iapr2025-2/train_annotated"
destination_path = "dataset_project_iapr2025-2/train_patches"
patches_from_coco(source_path, destination_path)

destination_path = "dataset_project_iapr2025-2/train_patches"
patches_from_coco(source_path, destination_path)

In [7]:
import os
import shutil
import json
import cv2
import pandas as pd
from tqdm import tqdm
from torchvision.datasets import CocoDetection


def extract_patch(image_path, bbox, size=1400):
    x, y, w, h = [int(coord) for coord in bbox]
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Image not found: {image_path}")
    patch = img[y:y+h, x:x+w]
    return patch

def patches_from_coco(source, dest):
    annotations_file = os.path.join(source, "_annotations.coco.json")
    images_dir = source
    patches_dir = os.path.join(dest, "patches")

    if os.path.exists(patches_dir):
        shutil.rmtree(patches_dir)
    os.makedirs(patches_dir)

    data = json.load(open(annotations_file, "r"))

    id_to_label = {e["id"]: e["name"] for e in data["categories"]}
    id_to_images = {e["id"]: e["file_name"] for e in data["images"]}
    annotations = data["annotations"]

    df_labels = pd.DataFrame(columns=["name", "label", "image", "bbox"])

    for i, annotation in tqdm(enumerate(annotations), total=len(annotations)):
        image_id = annotation["image_id"]
        label_id = annotation["category_id"]
        bbox = annotation["bbox"]
        label = id_to_label[label_id]
        image_file = id_to_images[image_id]
        image_path = os.path.join(images_dir, image_file)

        try:
            patch = extract_patch(image_path, bbox)
            idx = str(i).zfill(3)
            patch_path = os.path.join(patches_dir, f"{idx}.jpg")
            cv2.imwrite(patch_path, patch)
            df_labels.loc[i] = [idx, label, image_file, bbox]
        except Exception as e:
            print(f"⚠️ Skipped patch {i} from image {image_file}: {e}")

    df_labels.to_csv(os.path.join(patches_dir, "labels.csv"), index=False)
    print(f"✅ Saved {len(df_labels)} patches and labels to {patches_dir}")

# Cell 2: Define paths and load annotation info
train_img_dir = "dataset_project_iapr2025_coco/train_annotated"
ann_path = os.path.join(train_img_dir, "_annotations.coco.json")

# Load COCO annotations and count classes
with open(ann_path, "r") as f:
    coco_json = json.load(f)
categories = coco_json["categories"]
num_classes = len(categories) + 1  # +1 for background
print("Detected classes:", [c["name"] for c in categories])
print("Total (with background):", num_classes)


# Run it on your dataset
source_path = "dataset_project_iapr2025_coco/train_annotated"
destination_path = "dataset_project_iapr2025_coco/train_patches"
patches_from_coco(source_path, destination_path)

Detected classes: ['objects', 'Amandina', 'Arabia', 'Comtesse', 'Creme_brulee', 'Jelly_Black', 'Jelly_Milk', 'Jelly_White', 'Noblesse', 'Noir_authentique', 'Passion_au_lait', 'Stracciatella', 'Tentation_noir', 'Triangolo']
Total (with background): 15


  0%|          | 0/584 [00:00<?, ?it/s]

100%|██████████| 584/584 [00:20<00:00, 29.19it/s]

✅ Saved 584 patches and labels to dataset_project_iapr2025_coco/train_patches/patches


In [3]:
def patches_to_ImageFolder(src, dest):
    # Load the CSV with patch labels
    labels_csv = os.path.join(src, "labels.csv")
    df = pd.read_csv(labels_csv)

    # Clear and recreate the destination folder
    if os.path.exists(dest):
        shutil.rmtree(dest)
    os.makedirs(dest, exist_ok=True)

    # Create subfolders and copy images
    for label in tqdm(df["label"].unique(), desc="Creating folders"):
        label_dir = os.path.join(dest, label)
        os.makedirs(label_dir, exist_ok=True)

        for _, row in df[df["label"] == label].iterrows():
            patch_name = f"{str(row['name']).zfill(3)}.jpg"
            src_file = os.path.join(src, patch_name)
            dest_file = os.path.join(label_dir, patch_name)

            if os.path.exists(src_file):
                shutil.copy(src_file, dest_file)
            else:
                print(f"⚠️ Missing file: {src_file}")

# Run it on your dataset
train_src = os.path.join("dataset_project_iapr2025_coco", "train_patches", "patches")
train_dest = os.path.join("dataset_project_iapr2025_coco", "train_patches", "folder_dataset")
patches_to_ImageFolder(train_src, train_dest)

Creating folders: 100%|██████████| 13/13 [00:00<00:00, 157.03it/s]


In [4]:
from torchvision.datasets import CocoDetection

def get_transform():
    return transforms.Compose([
        transforms.Resize((400, 600)),  # Resize to 1400x1400
        transforms.ToTensor(),  # Converts PIL image or ndarray to tensor
    ])

def collate_fn(batch):
    images, targets = zip(*batch)
    converted_targets = []
    for target in targets:
        boxes = torch.as_tensor([obj['bbox'] for obj in target], dtype=torch.float32)
        boxes[:, 2:] += boxes[:, :2]  # Convert [x,y,w,h] to [x1,y1,x2,y2]
        labels = torch.as_tensor([obj['category_id'] for obj in target], dtype=torch.int64)
        converted_targets.append({'boxes': boxes, 'labels': labels})
    return list(images), converted_targets

# Load dataset
full_dataset = CocoDetection(root=train_img_dir, annFile=ann_path, transform=get_transform())

# Optional: Split into train/val
train_len = int(0.9 * len(full_dataset))
val_len = len(full_dataset) - train_len
train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# just for now 
test_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
print("Train/Val split:", train_len, "/", val_len)
print("display me first label for train dataset")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Train/Val split: 81 / 9
display me first label for train dataset


# Patches 

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

# === 1. Load CSV and create label map ===
csv_path = "dataset_project_iapr2025_coco/train_patches/patches/labels.csv"
image_dir = "dataset_project_iapr2025_coco/train_patches/patches"

df = pd.read_csv(csv_path, dtype={'name': str})
class_names = sorted(df['label'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
df['class_idx'] = df['label'].map(class_to_idx)

# === 2. Train/val split ===
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['class_idx'], random_state=42)

# === 3. Define transform ===
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# === 4. Define dataset ===
class ChocolatePatchDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, row['name'] + '.jpg')
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['class_idx'], dtype=torch.long)
        return image, label

# === 5. Create datasets and loaders ===
train_dataset = ChocolatePatchDataset(train_df, image_dir, transform)
val_dataset = ChocolatePatchDataset(val_df, image_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# === 6. Check dataset size ===
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
# === 7. Check first label ===
print("First label in train dataset:", train_dataset[0][1].item())

Train dataset size: 467
Validation dataset size: 117
First label in train dataset: 5


### training for CNN

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

class_number = len(full_dataset.coco.cats)
model = SimpleCNN(input_shape=3, hidden_units=64, image_height=128, image_width=128, output_shape=class_number)
loss_fn = nn.CrossEntropyLoss()

#optimizer = torch.optim.SGD(params=model.parameters(), lr=0.05)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

trainer = Trainer(model=model,
                  model_name="SimpleCNN",
                      loss_fn=loss_fn,
                      optimizer=optimizer,
                      scheduler=scheduler,
                      num_epochs=100,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)

trainer.train()
#predictions = trainer.predict()
#print(predictions)
#trainer.save_model()
avg_loss, overall_accuracy, overall_f1, per_class_f1 = trainer.evaluate()
#print(f"1 predition: {predictions[0][0]}")
# print(f"Ground truth: {val_ds[0][1]}")
# print(f"Average Loss: {avg_loss}, Accuracy: {accuracy}%")

/Users/ewamiazga/miniconda3/envs/iapr_project/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1: 100%|██████████| 30/30 [00:02<00:00, 11.00it/s]


Epoch [1/100], Loss: 2.2355


Epoch 2: 100%|██████████| 30/30 [00:02<00:00, 12.08it/s]


Epoch [2/100], Loss: 1.5047


Epoch 3: 100%|██████████| 30/30 [00:02<00:00, 11.77it/s]


...

Epoch 52: 100%|██████████| 30/30 [00:05<00:00,  5.48it/s]


Epoch [52/100], Loss: 0.0124


Epoch 53: 100%|██████████| 30/30 [00:05<00:00,  5.45it/s]


Epoch [53/100], Loss: 0.0119


Epoch 54:  13%|█▎        | 4/30 [00:00<00:04,  5.42it/s]

In [9]:
avg_loss, overall_accuracy, overall_f1, per_class_f1 = trainer.evaluate()
print(f"Average Loss: {avg_loss}, Overall Accuracy: {overall_accuracy}%, Overall F1: {overall_f1}")
# print(f"Per Class F1: {per_class_f1}")
for i, f1 in enumerate(per_class_f1):
    print(f"Class {i}: {class_names[i]} - F1 Score: {f1}")


100%|██████████| 4/4 [00:00<00:00,  8.93it/s]

Average Loss: 1.0863257497549057, Overall Accuracy: 0.8717948717948718%, Overall F1: 0.8729920552932243
Class 0: Amandina - F1 Score: 0.9230769230769231
Class 1: Arabia - F1 Score: 0.9523809523809523
Class 2: Comtesse - F1 Score: 0.782608695652174
Class 3: Creme_brulee - F1 Score: 0.9333333333333333
Class 4: Jelly_Black - F1 Score: 0.9333333333333333
Class 5: Jelly_Milk - F1 Score: 1.0
Class 6: Jelly_White - F1 Score: 0.6666666666666666
Class 7: Noblesse - F1 Score: 0.9473684210526315
Class 8: Noir_authentique - F1 Score: 0.9
Class 9: Passion_au_lait - F1 Score: 0.9
Class 10: Stracciatella - F1 Score: 0.7368421052631579
Class 11: Tentation_noir - F1 Score: 0.8
Class 12: Triangolo - F1 Score: 0.9411764705882353


# Sameh

Prepare the dataset

In [2]:
# Cell 2: Define paths and load annotation info
train_img_dir = "dataset_project_iapr2025_coco/train_annotated"
ann_path = "dataset_project_iapr2025_coco/train_annotated/_annotations.coco.json"

# Load COCO annotations and exclude "objects" class
with open(ann_path, "r") as f:
    coco_json = json.load(f)

# ✅ Exclude 'objects' and assign label IDs starting from 1
categories = [cat for cat in coco_json["categories"] if cat["name"] != "objects"]
class_name_to_id = {cat["name"]: i + 1 for i, cat in enumerate(categories)}
num_classes = max(class_name_to_id.values()) + 1  # +1 for background class 0

# Display the class structure
print("Detected classes (excluding 'objects'):", list(class_name_to_id.keys()))
print("Total (with background):", num_classes)


Detected classes (excluding 'objects'): ['Amandina', 'Arabia', 'Comtesse', 'Creme_brulee', 'Jelly_Black', 'Jelly_Milk', 'Jelly_White', 'Noblesse', 'Noir_authentique', 'Passion_au_lait', 'Stracciatella', 'Tentation_noir', 'Triangolo']
Total (with background): 14


In [3]:
def get_transform():
    return T.Compose([
        T.ToTensor(),  # Converts PIL image to tensor
    ])

# This collate_fn works for batched Faster R-CNN inputs
def collate_fn(batch):
    return tuple(zip(*batch))

# Load full dataset (with all classes including "objects")
full_dataset = CocoDataset(
    root=train_img_dir,
    annotation=ann_path,
    transform=get_transform()
)

# Split full dataset into 90% train / 10% validation
total_len = len(full_dataset)
train_len = int(0.9 * total_len)
val_len = total_len - train_len

train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_len, val_len])

### TODO: Add test_loader if needed
test_image_dir = "dataset_project_iapr2025/test"
unlabeled_dataset = UnlabeledImageFolder(test_image_dir)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(unlabeled_dataset, batch_size=1, shuffle=False)
#test_loader = unlabeled_dataset

print(f"Loaded full dataset: {total_len} images -> {train_len} train / {val_len} val")

Loaded full dataset: 90 images -> 81 train / 9 val


Train the model 

In [ ]:
model = LightFasterRCNNMobileNetV3(num_classes=num_classes)
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
epochs = 150

trainer = Trainer(model=model,
                  model_name="MobileNetV3",
                      optimizer=optimizer,
                      num_epochs=epochs,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)

#trainer.train()
#avg_loss, overall_accuracy, overall_f1, per_class_f1 = trainer.evaluate()

Total parameters:         8690740
Backbone (MobileNet):     1371344
RPN:                      597790
ROI Heads (Box Head):     6721606


NameError: name 'ć' is not defined

In [ ]:
avg_loss, overall_accuracy, overall_f1, per_class_f1 = trainer.evaluate()
print(f"Average Loss: {avg_loss}, Overall Accuracy: {overall_accuracy}%, Overall F1: {overall_f1}")
# print(f"Per Class F1: {per_class_f1}")
for i, f1 in enumerate(per_class_f1):
    print(f"Class {i}: {class_names[i]} - F1 Score: {f1}")

Test the model

In [ ]:
predictions = trainer.predict()

print(f"1 predition: {predictions[0][0]}")

AttributeError: 'tuple' object has no attribute 'to'

Save the model

In [ ]:
trainer.save_model()